# 40 - Pre-Train ResNet18 pada FER2013

**Tujuan:** Pre-train ResNet18 di FER2013 (35K gambar emosi) sebelum fine-tune ke dataset sendiri.
Ini memberikan fitur yang lebih relevan (ekspresi wajah) dibanding ImageNet (kucing, mobil, dll).

**Pipeline:** ImageNet -> FER2013 -> Dataset sendiri (two-stage TL)

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import EmotionCNNTransfer
from training.utils import train_model, full_evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

FER_DIR = PROJECT_ROOT / "data" / "benchmark" / "fer2013_prepared"
OUTPUT_DIR = PROJECT_ROOT / "models" / "pretrained"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 10
LR = 0.0001
NUM_CLASSES = 7
EMOTIONS = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"]
print("Setup complete.")

Device: cuda
GPU: Tesla T4
Setup complete.


## Load FER2013

In [2]:
# Load FER2013
train_img = np.load(FER_DIR / "X_train_images.npy")
train_y = np.load(FER_DIR / "y_train.npy")
val_img = np.load(FER_DIR / "X_val_images.npy")
val_y = np.load(FER_DIR / "y_val.npy")

print(f"Train: {len(train_y)} | Val: {len(val_y)}")
print(f"Distribution: {dict(sorted(Counter(train_y.tolist()).items()))}")

# DataLoaders
def make_loader(images, labels, batch_size=32, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    y_t = torch.from_numpy(labels).long()
    return DataLoader(TensorDataset(img_t, y_t), batch_size=batch_size,
                      shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(train_img, train_y, BATCH_SIZE)
val_loader = make_loader(val_img, val_y, BATCH_SIZE, False)
print("Data loaded.")

Train: 25839 | Val: 2870
Distribution: {0: 4479, 1: 6419, 2: 4359, 3: 3635, 4: 3664, 5: 394, 6: 2889}
Data loaded.


## Pre-Train ResNet18 on FER2013

In [3]:
# Pre-train ResNet18 (ImageNet -> FER2013)
model = EmotionCNNTransfer(num_classes=NUM_CLASSES, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-7)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Pre-training on FER2013...")

history, best_epoch = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, model_type="cnn", epochs=EPOCHS, patience=PATIENCE,
    save_path=str(OUTPUT_DIR / "resnet18_fer2013.pth"))

print(f"\nBest epoch: {best_epoch}")
print(f"Saved: {OUTPUT_DIR / 'resnet18_fer2013.pth'}")

Parameters: 11,310,151
Pre-training on FER2013...
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.2422     0.5374     0.9932    0.6338   0.5273   0.000100  (92.3s)


     2      0.9417     0.6564     0.9291    0.6523   0.6105   0.000100  (88.9s)


     3      0.7346     0.7379     0.9268    0.6676   0.6482   0.000100  (88.4s)


     4      0.5127     0.8229     1.0324    0.6453   0.6352   0.000100  (87.8s)


...

    26      0.0218     0.9928     1.8082    0.6812   0.6763   0.000050  (87.0s)


    27      0.0191     0.9936     1.8081    0.6693   0.6572   0.000050  (87.0s)


    28      0.0098     0.9968     1.7558    0.6763   0.6654   0.000025  (87.1s)


    29      0.0069     0.9973     1.7866    0.6815   0.6704   0.000025  (87.1s)


    30      0.0059     0.9978     1.8133    0.6833   0.6748   0.000025  (87.0s)

Best: epoch 21, val_acc=0.6864, val_f1=0.6799
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/pretrained/resnet18_fer2013.pth

Best epoch: 21
Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/pretrained/resnet18_fer2013.pth


## Evaluate on FER2013 Test Set

In [4]:
# Evaluate
test_img = np.load(FER_DIR / "X_test_images.npy") if (FER_DIR / "X_test_images.npy").exists() else val_img
test_y = np.load(FER_DIR / "y_test.npy") if (FER_DIR / "y_test.npy").exists() else val_y
test_loader = make_loader(test_img, test_y, BATCH_SIZE, False)

model.load_state_dict(torch.load(OUTPUT_DIR / "resnet18_fer2013.pth", map_location=device, weights_only=True))
results = full_evaluation(model, test_loader, criterion, device, "cnn", EMOTIONS)

print(f"\nFER2013 Test Results:")
print(f"  Accuracy:  {results['test_accuracy']:.4f}")
print(f"  Macro F1:  {results['test_macro_f1']:.4f}")
print(f"  W-F1:      {results['test_weighted_f1']:.4f}")
print(f"\nPre-trained weights ready at: {OUTPUT_DIR / 'resnet18_fer2013.pth'}")

Test Loss: 1.6362
Test Accuracy: 0.6783
Test Macro F1: 0.6585
Test Weighted F1: 0.6780

Classification Report:
              precision    recall  f1-score   support

     neutral       0.64      0.66      0.65      1233
       happy       0.87      0.87      0.87      1774
         sad       0.52      0.60      0.56      1247
       angry       0.61      0.57      0.59       958
     fearful       0.56      0.48      0.52      1024
   disgusted       0.74      0.54      0.62       111
   surprised       0.80      0.81      0.80       831

    accuracy                           0.68      7178
   macro avg       0.68      0.65      0.66      7178
weighted avg       0.68      0.68      0.68      7178


FER2013 Test Results:
  Accuracy:  0.6783
  Macro F1:  0.6585
  W-F1:      0.6780

Pre-trained weights ready at: /home/bs000716/MOTHER-TANK/TRAIN/models/pretrained/resnet18_fer2013.pth
